# Imaged FOVs — live acquisition progress map

Watches one round and fills in each FOV as its image file appears, to see
where on the coverslip the microscope is without opening Steve.

Background: the Steve mosaic if the boundaries came from one (02a, needs
`data/mosaic10x/*.msc`), else the tissue outline drawn by 02b.

**Colors:** gray outline = not imaged yet, filled = imaged. A FOV is flagged
(`WARNING_COLOR`) when its `.off` file has `good-offset` = 1 in fewer than
`MIN_GOOD_OFFSET_FRAMES` frames. The default 1 flags only FOVs where the focus
lock was never found; losing it partway up the z-sweep is normal. Raise it to
also flag brief locks. A heuristic, not Dave's "Check Focus Lock" warning.

**Usage:** run every cell, leave the last one running while the round images.
Interrupt the kernel to stop; the last view is saved to `figures/`.

## 1 — Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, clear_output

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/during_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.configs    import get_fov_geometry
from MERci.acquisition.positions  import (
    resolve_boundaries_source_dir, discover_boundary_files,
    load_boundary_polygon, load_hole_polygons,
)
from MERci.acquisition.mosaic     import load_or_build_mosaic_canvas_cached
from MERci.analysis.stage_z       import focus_lock_summary_for_fov
from MERci.analysis.imaged_fovs   import round_progress, detect_active_round
from MERci.plots.experiment_plots import get_merci_figures_dir

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX  = ".zarr"   # same as HAL writes
NOTEBOOK_NAME = "imaged_fovs"   # cache and figures folder name

MICROSCOPE = "MF3"   # camera geometry from data/configs/merlin/microscope/
OBJECTIVE  = "60X"   # "60X" | "40X" (40X on ST2 only)
pixel_size_um, image_size_px = get_fov_geometry(MICROSCOPE, OBJECTIVE)
fov_size_um = pixel_size_um * image_size_px

# Round to watch. None = the round being written now, else the next one to
# start. Or an imaging_round number.
ROUND_ID = None

POLL_INTERVAL_SEC = 5     # must be well under the time to acquire one FOV
MAX_RUNTIME_MIN    = 240  # stop polling after this many minutes

UNIMAGED_EDGECOLOR = "0.55"
IMAGED_COLOR       = "gold"
WARNING_COLOR      = "red"   # focus-lock flag

# Flag a FOV with good-offset == 1 in fewer than this many frames (1 = never
# locked; e.g. 5 also flags brief locks).
MIN_GOOD_OFFSET_FRAMES = 1

# Mosaic assembly: same values as in 02a, or the image won't line up.
MOSAIC_KEEP_OBJECTIVES  = ["10x"]
MOSAIC_WORKING_PIXEL_UM = 5.0

config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

CACHE_DIR = config.analysis_dir / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
# A live view, not a QC figure.
FIGURES_DIR = get_merci_figures_dir(SAMPLE_DIR, "during_imaging", NOTEBOOK_NAME)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample name  : {SAMPLE_NAME}")
print(f"FOVs (total) : {meta.n_fovs}")
print(f"Rounds       : {sorted(meta.rounds)}")
print(f"FOV size     : {fov_size_um:.1f} um  ({MICROSCOPE}, {OBJECTIVE})")

## 3 — Pick the round to watch

With `ROUND_ID = None`, finds the round being written (file checks only, no
image reads). Between rounds (during fluidics), picks the next round, so the
view is ready when it starts.

In [ ]:
if ROUND_ID is None:
    ROUND_ID = detect_active_round(config, meta)
    print(f"Auto-detected ROUND_ID = {ROUND_ID} (override the parameter above to pick a different round)")
else:
    print(f"Watching explicit ROUND_ID = {ROUND_ID}")

ROUND_SERIES = meta.series_for_round(ROUND_ID)
if not ROUND_SERIES:
    raise ValueError(f"round_info.csv has no series for imaging_round={ROUND_ID}")

n_imaged_now, _ = round_progress(ROUND_ID, config, meta)
print(f"Round {ROUND_ID}: {n_imaged_now}/{meta.n_fovs} FOV(s) already imaged.")

## 4 — Load the background (real mosaic photo, else schematic outline)

Same boundary and mosaic code as 02b, with the same canvas cache. No mosaic
unless `BOUNDARY_SOURCE == "from_mosaic"` and a `.msc` exists.

In [ ]:
POSITIONS_DIR = SAMPLE_DIR / "positions"
BOUNDARY_DIR, BOUNDARY_SOURCE = resolve_boundaries_source_dir(POSITIONS_DIR, None)

boundaries, MODE = ([], None)
boundary_polys   = []
holes            = []
mosaic_canvas    = None

if BOUNDARY_DIR is not None and BOUNDARY_DIR.exists():
    boundaries, MODE = discover_boundary_files(BOUNDARY_DIR)
    boundary_polys    = [load_boundary_polygon(b.path) for b in boundaries]
    holes             = load_hole_polygons(BOUNDARY_DIR)

MOSAIC_DIR = SAMPLE_DIR / "data" / "mosaic10x"
msc_candidates = sorted(MOSAIC_DIR.glob("*.msc")) if BOUNDARY_SOURCE == "from_mosaic" else []

if msc_candidates:
    mosaic_canvas, was_cached, n_tiles = load_or_build_mosaic_canvas_cached(
        msc_candidates[0], MOSAIC_KEEP_OBJECTIVES, MOSAIC_WORKING_PIXEL_UM, CACHE_DIR)
    if was_cached:
        print(f"Loaded cached mosaic canvas: {CACHE_DIR / 'mosaic_canvas.npz'}")
    else:
        print(f"Assembled + cached mosaic canvas from {n_tiles} tile(s): {CACHE_DIR / 'mosaic_canvas.npz'}")
    print(f"Background  : real mosaic photo ({mosaic_canvas.image.shape[1]}x{mosaic_canvas.image.shape[0]} px "
          f"at {mosaic_canvas.pixel_size_um:.2f} um/px)")
else:
    print("Background  : schematic tissue-boundary outline "
          "(no from_mosaic boundaries / .msc file found)")

## 5 — Build the static base plot (one square per planned FOV)

In [ ]:
fov_ids = sorted(meta.fovs)
patch_of = {}

if mosaic_canvas is not None:
    half_px = (fov_size_um / 2) / mosaic_canvas.pixel_size_um

    xs_um = np.array([meta.fovs[f].position[0] for f in fov_ids])
    ys_um = np.array([meta.fovs[f].position[1] for f in fov_ids])
    px, py = mosaic_canvas.to_px(xs_um, ys_um)
    margin_px = 5 * (fov_size_um / mosaic_canvas.pixel_size_um)
    x_min, x_max = px.min() - margin_px, px.max() + margin_px
    y_min, y_max = py.min() - margin_px, py.max() + margin_px

    step_px      = fov_size_um / mosaic_canvas.pixel_size_um
    inch_per_fov = 0.35
    fig_w = float(np.clip((x_max - x_min) / step_px * inch_per_fov, 8, 60))
    fig_h = float(np.clip((y_max - y_min) / step_px * inch_per_fov, 8, 60))

    covered_vals = mosaic_canvas.image[mosaic_canvas.covered]
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.imshow(mosaic_canvas.image, cmap="gray",
              vmin=np.percentile(covered_vals, 1), vmax=np.percentile(covered_vals, 99))

    for poly in boundary_polys:
        bx, by = mosaic_canvas.to_px(*poly.exterior.xy)
        ax.plot(bx, by, "-", lw=1.0, color="tab:blue")
    for hole in holes:
        hx, hy = mosaic_canvas.to_px(*hole.exterior.xy)
        ax.plot(hx, hy, "r--", lw=0.8)

    for fov_id, cx, cy in zip(fov_ids, px, py):
        patch_of[fov_id] = mpatches.Rectangle(
            (cx - half_px, cy - half_px), 2 * half_px, 2 * half_px,
            lw=0.4, edgecolor=UNIMAGED_EDGECOLOR, facecolor="none",
        )
        ax.add_patch(patch_of[fov_id])

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_max, y_min)   # imshow's default origin puts row 0 at the top
    ax.set_xlabel("canvas column (px)")
    ax.set_ylabel("canvas row (px)")
else:
    half = fov_size_um / 2
    fig, ax = plt.subplots(figsize=(8, 7))

    for poly in boundary_polys:
        ax.plot(*poly.exterior.xy, "-", lw=1.0, color="tab:blue")
    for hole in holes:
        ax.fill(*hole.exterior.xy, color="0.85", alpha=0.6)
        ax.plot(*hole.exterior.xy, "k--", lw=0.8)

    for fov_id in fov_ids:
        x, y = meta.fovs[fov_id].position
        patch_of[fov_id] = mpatches.Rectangle(
            (x - half, y - half), fov_size_um, fov_size_um,
            lw=0.4, edgecolor=UNIMAGED_EDGECOLOR, facecolor="none",
        )
        ax.add_patch(patch_of[fov_id])

    ax.invert_yaxis()
    ax.axis("equal")
    ax.set_xlabel("Stage X (um)")
    ax.set_ylabel("Stage Y (um)")

ax.set_title(f"Imaged FOVs — {SAMPLE_NAME}, round {ROUND_ID}")
fig.tight_layout()
display(fig)

## 6 — Live loop: color a square in as its FOV is imaged

Interrupt the kernel to stop — the figure keeps its last drawn state and is
saved to `figures/` either way.

In [ ]:
imaged      = set()
warned      = set()
off_checked = set()   # fov_ids whose .off file gave a final verdict

start_time = time.time()
poll_count = 0

try:
    while len(imaged) < len(fov_ids):
        if (time.time() - start_time) > MAX_RUNTIME_MIN * 60:
            print(f"Stopping: MAX_RUNTIME_MIN={MAX_RUNTIME_MIN} exceeded with "
                  f"{len(imaged)}/{len(fov_ids)} FOVs imaged.")
            break

        poll_count += 1
        newly_imaged = []
        for fov_id in fov_ids:
            if fov_id in imaged:
                continue
            paths = [s.resolve_path(fov_id, config.image_suffix) for s in ROUND_SERIES]
            existing = [p for p in paths if p.exists()]
            if existing:
                imaged.add(fov_id)
                newly_imaged.append((fov_id, existing[0]))

        for fov_id, image_path in newly_imaged:
            patch_of[fov_id].set_facecolor(IMAGED_COLOR)
            patch_of[fov_id].set_edgecolor(IMAGED_COLOR)
            patch_of[fov_id].set_alpha(0.6)

        # .off files can lag the image file; retry FOVs not classified yet.
        pending = [f for f in imaged if f not in off_checked]
        for fov_id in pending:
            paths = [s.resolve_path(fov_id, config.image_suffix) for s in ROUND_SERIES]
            existing = [p for p in paths if p.exists()]
            summary = focus_lock_summary_for_fov(existing[0]) if existing else None
            if summary is None:
                continue   # .off not written yet -- retry next poll
            off_checked.add(fov_id)
            n_good_frames = summary["n_frames"] - summary["n_bad_frames"]
            flagged = n_good_frames < MIN_GOOD_OFFSET_FRAMES
            if flagged:
                warned.add(fov_id)
                patch_of[fov_id].set_facecolor(WARNING_COLOR)
                patch_of[fov_id].set_edgecolor(WARNING_COLOR)
                patch_of[fov_id].set_alpha(0.8)

        elapsed = time.time() - start_time
        rate    = len(imaged) / elapsed if elapsed > 0 else 0.0
        eta_sec = (len(fov_ids) - len(imaged)) / rate if rate > 0 else float("nan")
        eta_str = f"{eta_sec / 60:.1f} min" if rate > 0 else "n/a"

        clear_output(wait=True)
        display(fig)
        print(f"poll #{poll_count} | elapsed {elapsed / 60:.1f} min | "
              f"imaged {len(imaged)}/{len(fov_ids)} ({100 * len(imaged) / len(fov_ids):.1f}%) | "
              f"warnings: {len(warned)} | ETA {eta_str} | next check in {POLL_INTERVAL_SEC}s")

        if len(imaged) < len(fov_ids):
            time.sleep(POLL_INTERVAL_SEC)
except KeyboardInterrupt:
    print(f"Stopped by user at {len(imaged)}/{len(fov_ids)} FOVs imaged.")

fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.progress.png", dpi=150)
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.progress.png'}")
